# Silver Layer — All Tables Pipeline

Processes all Bronze tables into Silver Delta tables incrementally:
- **High-Watermark Incremental Read**: Only reads records newer than the latest timestamp in Silver
- **Domain Cleaning & Standardization**: Normalizes strings, parses dates/timestamps, standardizes codes
- **Deduplication**: Retains the latest record per primary key
- **Delta MERGE (Upsert)**: Seamlessly merges new and updated records into the target Silver table
- **Full Table Coverage**: Handles all 15 tables across EMR, Laboratory, Pharmacy, and Billing domains

In [ ]:
from datetime import datetime
import pandas as pd

from config.silver_config import (
    SILVER_TABLE_CONFIG,
    CATALOG,
    SILVER_SCHEMA,
    get_bronze_table,
    get_silver_table,
)
from src.silver.transforms import TRANSFORM_FUNCTIONS
from src.silver.deduplication import deduplicate_latest
from src.silver.writer import merge_to_silver

In [ ]:
# Create the Silver schema if it does not exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")
print(f"Silver schema verified: {CATALOG}.{SILVER_SCHEMA}")

In [ ]:
# Filter to a specific table or process ALL (e.g. "ALL" or "patients")
TABLE_FILTER = "ALL"

tables_to_process = (
    list(SILVER_TABLE_CONFIG.keys())
    if TABLE_FILTER == "ALL"
    else [TABLE_FILTER]
)

print(f"Starting Silver processing at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Tables to process ({len(tables_to_process)}): {', '.join(tables_to_process)}")

In [ ]:
results = []

for table_name in tables_to_process:
    config = SILVER_TABLE_CONFIG[table_name]
    transform_fn = TRANSFORM_FUNCTIONS[table_name]
    bronze_tbl = get_bronze_table(table_name)
    silver_tbl = get_silver_table(table_name)

    print("=" * 70)
    print(f"Processing: {table_name}")
    print(f"  Source:      {bronze_tbl}")
    print(f"  Target:      {silver_tbl}")
    print(f"  Primary Key: {config['primary_key']}")
    print("=" * 70)

    start_time = datetime.now()

    try:
        result = merge_to_silver(
            spark=spark,
            bronze_table=bronze_tbl,
            silver_table=silver_tbl,
            primary_key=config["primary_key"],
            transform_fn=transform_fn,
            deduplicate_fn=deduplicate_latest,
            dedup_partition_cols=config["partition_cols"],
            dedup_order_col=config["order_col"]
        )

        elapsed = (datetime.now() - start_time).total_seconds()

        results.append({
            "table": table_name,
            "status": result["status"],
            "rows_processed": result["rows_processed"],
            "duration_sec": round(elapsed, 2),
            "error": None
        })

        print(f"  -> SUCCESS: {result['status']} | {result['rows_processed']} rows in {elapsed:.2f}s")

    except Exception as e:
        elapsed = (datetime.now() - start_time).total_seconds()

        results.append({
            "table": table_name,
            "status": "FAILED",
            "rows_processed": 0,
            "duration_sec": round(elapsed, 2),
            "error": str(e)
        })

        print(f"  -> FAILED: {table_name}")
        print(f"  -> Error: {e}")

In [ ]:
# Summary report
summary_df = pd.DataFrame(results)
total_rows = summary_df["rows_processed"].sum()
failed_count = (summary_df["status"] == "FAILED").sum()
success_count = len(summary_df) - failed_count

print("\n" + "=" * 70)
print("SILVER PROCESSING SUMMARY")
print("=" * 70)
print(f"Total tables:    {len(summary_df)}")
print(f"Successful:      {success_count}")
print(f"Failed:          {failed_count}")
print(f"Total rows:      {total_rows}")
print("=" * 70)

display(summary_df)